In [30]:
import sys
sys.path.insert(0, "../src")


from pyspark.sql import SparkSession, functions as F
import main, m1, o2
import importlib
importlib.reload(o2)
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("dev-o2")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

In [31]:
trips = main.load_trips(
    spark, "../databases", [2025], 1.0
)

trips_clean = m1.clean(
    m1.flag_invalid(trips, years=[2025])
)

zones = main.load_zones(
    spark, "../databases/taxi_zone_lookup.csv"
)

print("Clean trips:", trips_clean.count())

Clean trips: 8482202


In [32]:
print("Zones columns:", zones.columns)
zones.printSchema()


Zones columns: ['LocationID', 'Borough', 'Zone', 'service_zone']
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [33]:
import importlib
importlib.reload(o2)
flows = o2.build_monthly_flows(trips_clean, zones)

flows.printSchema()
flows.show(10, truncate=False)

root
 |-- DOLocationID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- trips: long (nullable = false)
 |-- outbound_trips: long (nullable = true)
 |-- route_share: double (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- dropoff_borough: string (nullable = true)

+------------+------------+----+-----+-----+--------------+---------------------+-----------------------+--------------+-----------------------------+---------------+
|DOLocationID|PULocationID|year|month|trips|outbound_trips|route_share          |pickup_zone            |pickup_borough|dropoff_zone                 |dropoff_borough|
+------------+------------+----+-----+-----+--------------+---------------------+-----------------------+--------------+-----------------------------+---------------+
|211         |144        

In [34]:
share_check = (
    flows
    .groupBy("year", "month", "PULocationID")
    .agg(
        F.sum("route_share").alias("sum_route_shares")
    )
)

share_check.select(
    F.min("sum_route_shares").alias("min_share_sum"),
    F.max("sum_route_shares").alias("max_share_sum")
).show()

+-----------------+------------------+
|    min_share_sum|     max_share_sum|
+-----------------+------------------+
|0.999999999999998|1.0000000000000013|
+-----------------+------------------+



In [35]:
top20 = o2.top_routes(flows, n=20)

top20.orderBy(
    "year",
    "month",
    "pickup_borough",
    "route_rank"
).show(50, truncate=False)

+------------+------------+----+-----+-----+--------------+--------------------+--------------------------------+--------------+-------------------------+---------------+----------+
|DOLocationID|PULocationID|year|month|trips|outbound_trips|route_share         |pickup_zone                     |pickup_borough|dropoff_zone             |dropoff_borough|route_rank|
+------------+------------+----+-----+-----+--------------+--------------------+--------------------------------+--------------+-------------------------+---------------+----------+
|75          |51          |2025|1    |25   |337           |0.07418397626112759 |Co-Op City                      |Bronx         |East Harlem South        |Manhattan      |1         |
|197         |254         |2025|1    |24   |276           |0.08695652173913043 |Williamsbridge/Olinville        |Bronx         |Richmond Hill            |Queens         |2         |
|42          |168         |2025|1    |22   |323           |0.06811145510835913 |Mott Haven

In [36]:
reversals = o2.share_reversals(flows)

reversals.orderBy(
    "year",
    "pickup_borough",
    "PULocationID",
    "DOLocationID"
).show(50, truncate=False)

+----+------------+------------+-----------+--------------+------------+---------------+---------+---------+---------+---------+---------+---------+--------------+
|year|PULocationID|DOLocationID|pickup_zone|pickup_borough|dropoff_zone|dropoff_borough|jan_share|mar_share|jul_share|sep_share|q1_change|q3_change|share_reversal|
+----+------------+------------+-----------+--------------+------------+---------------+---------+---------+---------+---------+---------+---------+--------------+
+----+------------+------------+-----------+--------------+------------+---------------+---------+---------+---------+---------+---------+---------+--------------+



In [38]:
print("Monthly route rows:", flows.count())
print("Top-20 rows:", top20.count())
print("Share reversals:", reversals.count())

o2.write(flows, top20, reversals)

Monthly route rows: 74840
Top-20 rows: 330
Share reversals: 0


In [29]:
flows.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (53)
+- Project (52)
   +- BroadcastHashJoin LeftOuter BuildRight (51)
      :- Project (46)
      :  +- BroadcastHashJoin LeftOuter BuildRight (45)
      :     :- Project (40)
      :     :  +- SortMergeJoin Inner (39)
      :     :     :- Sort (18)
      :     :     :  +- Exchange (17)
      :     :     :     +- HashAggregate (16)
      :     :     :        +- Exchange (15)
      :     :     :           +- HashAggregate (14)
      :     :     :              +- Union (13)
      :     :     :                 :- Project (4)
      :     :     :                 :  +- Project (3)
      :     :     :                 :     +- Filter (2)
      :     :     :                 :        +- Scan parquet  (1)
      :     :     :                 :- Project (8)
      :     :     :                 :  +- Project (7)
      :     :     :                 :     +- Filter (6)
      :     :     :                 :        +- Scan parquet  (5)
      :     :     :           